In [ ]:
import re
import csv
import time
import requests
from bs4 import BeautifulSoup

# --- Config: what we're scraping -------------------------------------------
BASE = "https://www.dbl.id/match/schedule"
PARAMS_BASE = {"season_year": 2026, "series_id": 2, "region_id": 2}  # East Java Series, South Region
HEADERS = {"User-Agent": "Mozilla/5.0 (research script; contact: you@example.com)"}

# Known season dates as of Sep 2026 (discovered by inspecting the schedule
# page's date-tab navigation, which lists every date across every season).
# The scraper also re-discovers dates live from the page, so this list is
# just a fallback / starting point and will auto-extend as new seasons run.
# CHANGED: dates we've specifically confirmed (by manually checking dbl.id)
# should have real games — guarantees a debug snippet gets saved for these
# even if the per-year cap gets used up by earlier dates first.
MUST_DEBUG_DATES = {"2022-09-06", "2026-08-14"}

FALLBACK_DATES = {
    2019: ["2019-08-31","2019-09-01","2019-09-02","2019-09-03","2019-09-04","2019-09-05",
           "2019-09-06","2019-09-07","2019-09-08","2019-09-09","2019-09-10","2019-09-11",
           "2019-09-14","2019-09-15","2019-09-17","2019-09-18","2019-09-20"],
    2022: ["2022-08-30","2022-08-31","2022-09-01","2022-09-02","2022-09-03","2022-09-04",
           "2022-09-05","2022-09-06","2022-09-07","2022-09-08","2022-09-09","2022-09-10",
           "2022-09-13","2022-09-14","2022-09-15","2022-09-16","2022-09-17","2022-09-19"],
    2023: ["2023-08-03","2023-08-04","2023-08-05","2023-08-06","2023-08-07","2023-08-08",
           "2023-08-09","2023-08-10","2023-08-11","2023-08-12","2023-08-13","2023-08-14",
           "2023-08-15","2023-08-17","2023-08-18","2023-08-19","2023-08-20","2023-08-21","2023-08-23"],
    2024: ["2024-09-04","2024-09-05","2024-09-06","2024-09-07","2024-09-08","2024-09-09",
           "2024-09-10","2024-09-11","2024-09-12","2024-09-13","2024-09-14","2024-09-15",
           "2024-09-16","2024-09-17","2024-09-18","2024-09-20","2024-09-21","2024-09-22",
           "2024-09-23","2024-09-24","2024-09-26"],
    2025: ["2025-08-12","2025-08-13","2025-08-14","2025-08-15","2025-08-16","2025-08-17",
           "2025-08-18","2025-08-19","2025-08-20","2025-08-21","2025-08-22","2025-08-23",
           "2025-08-24","2025-08-25","2025-08-26","2025-08-28","2025-08-29","2025-08-30",
           "2025-08-31","2025-09-06","2025-09-07"],
    2026: ["2026-08-13","2026-08-14","2026-08-15","2026-08-16","2026-08-17","2026-08-18",
           "2026-08-19","2026-08-20","2026-08-21","2026-08-22","2026-08-23","2026-08-24",
           "2026-08-25","2026-08-26","2026-08-27","2026-08-29","2026-08-30","2026-08-31",
           "2026-09-01","2026-09-02","2026-09-04"],
}


def discover_all_dates(session):
    """Try to find extra dates live (in case a new season/day has been added
    since this script was written), but NEVER let this override the known,
    verified FALLBACK_DATES list below — dbl.id's date-tab navigation is
    populated dynamically, so a plain `requests` GET (no JS execution) often
    sees a stripped-down page and only picks up a handful of stray date
    links. Discovered dates are only ever added on top, not substituted in.
    """
    known = set()
    for yr_dates in FALLBACK_DATES.values():
        known.update(yr_dates)

    try:
        r = session.get(BASE, params=PARAMS_BASE, headers=HEADERS, timeout=20)
        r.raise_for_status()
        discovered = set(re.findall(r"date=(\d{4}-\d{2}-\d{2})", r.text))
        new_ones = discovered - known
        if new_ones:
            print(f"Found {len(new_ones)} additional date(s) not in the "
                  f"hardcoded list: {sorted(new_ones)}")
        else:
            print("Live discovery found nothing new (expected — dbl.id's "
                  "date tabs are JS-rendered and won't show up here). "
                  f"Using the {len(known)} known verified dates.")
        known.update(new_ones)
    except Exception as e:
        print(f"Live discovery failed ({e}); using the {len(known)} known "
              "verified dates only.")

    return sorted(known)


def parse_day(html, date_str):
    """Extract every game on one schedule/results day-page.

    dbl.id doesn't expose a JSON API — this is plain HTML scraping. Each
    game on the page is represented by a "card" of text containing the two
    team names, the two scores, and a meta line like
    "Boys • 13:00 • GOR Ken Arok Malang • Group Stage", plus a link to that
    game's own detail page (/match/detail/<id>), which is what we anchor on.
    """
    soup = BeautifulSoup(html, "html.parser")
    games = []

    # Step 1: find every "Game Detail" link on the page — one per game.
    # We use these as anchors and walk UP the DOM tree from each link until
    # we land on a container that holds the whole card's text (teams,
    # scores, category, time, venue, round) — not just the link itself.
    for link in soup.select('a[href*="/match/detail/"]'):
        href = link.get("href", "")
        m = re.search(r"/match/detail/(\d+)", href)
        if not m:
            continue
        game_id = m.group(1)

        # Walk up to 6 parent levels, stopping once the container's text
        # looks "big enough" to be the whole card (has the • separators AND
        # a clock-style time in it) — not just the link by itself.
        card = link
        for _ in range(6):
            if card.parent is None:
                break
            card = card.parent
            text = card.get_text(" | ", strip=True)
            if text.count("|") >= 6 and re.search(r"\d{1,2}:\d{2}", text):
                break

        # Step 2: pull the card's text out as separate lines and classify
        # each line by shape: all-digits = a score, contains "•" = metadata line.
        # Remaining non-empty lines are assumed to be team names.
        text = card.get_text("\n", strip=True)
        lines = [l for l in text.split("\n") if l.strip()]

        nums = []
        meta_line = ""
        potential_names = [] # To hold potential team names

        for l in lines:
            if re.fullmatch(r"\d{1,3}", l):
                nums.append(l)
            elif "•" in l:
                meta_line = l
            else:
                potential_names.append(l)

        # Filter out very short lines from potential_names that might be artifacts
        names = [l for l in potential_names if len(l) > 2]

        # If we didn't find the shape we expect, skip this card rather than
        # guessing wrong data into the dataset.
        if len(nums) < 2 or len(names) < 2 or not meta_line:
            continue

        # Step 3: split the "Category • Time • Venue • Round" meta line.
        parts = [p.strip() for p in meta_line.split("•")]
        category = parts[0] if len(parts) > 0 else ""
        time_wib = next((p for p in parts if re.search(r"\d{1,2}:\d{2}", p)), "")
        venue = parts[-2] if len(parts) >= 2 else ""
        round_name = parts[-1] if len(parts) >= 1 else ""

        games.append({
            "game_id": game_id,
            "date": date_str,
            "season_year": date_str[:4],
            "series": "East Java Series",
            "region": "South Region",
            "category": category,
            "round": round_name,
            "venue": venue,
            "time_wib": time_wib,
            "home_team": names[0],
            "away_team": names[1],
            "home_score": int(nums[0]),
            "away_score": int(nums[1]),
            "source_url": f"https://www.dbl.id/match/detail/{game_id}",
        })

    # Step 4: de-duplicate. dbl.id's page markup repeats each game's card
    # twice (once for a "grid" view, once for a "list" view) — both contain
    # a /match/detail/ link with the same game_id, so without this we'd
    # double-count every game.
    seen, unique = set(), []
    for g in games:
        if g["game_id"] not in seen:
            seen.add(g["game_id"])
            unique.append(g)

    # Step 5: Boys 5-on-5 only — drop Girls and any 3x3 category.
    unique = [g for g in unique if g["category"].strip().lower() == "boys"]
    return unique


def main():
    # CHANGED: version banner — prints first, so you can confirm at a
    # glance whether you're actually running this updated file (with the
    # rate-limit backoff/retry logic below) and not an older saved copy.
    print("=== scrape_dbl_south_region.py — v5 (per-year + guaranteed debug snippets) ===\n")

    # Step 1: figure out which dates to scrape (all known match-days across
    # every season, see discover_all_dates() above).
    session = requests.Session()
    all_dates = discover_all_dates(session)
    print(f"Found {len(all_dates)} match days to scrape.")

    # Step 2: fetch and parse each date's page, one at a time.
    all_games = []
    zero_game_days = []
    seen_response_hashes = {}
    snippets_saved_by_year = {}  # CHANGED: cap per-year, not globally — the
    # global cap of 3 last time got used up entirely on 2019 dates, leaving
    # zero diagnostic info for 2022/2025/2026, which is exactly what we
    # needed to see (e.g. 2022-09-06, confirmed to have a real game).  # CHANGED: content hash -> first date_str that produced it
    for i, date_str in enumerate(all_dates, 1):
        year = int(date_str[:4])

        # CHANGED: dropped the two-variant ("Variant A/B") retry from the
        # last version — it was based on a wrong theory (that the current
        # season needed season_year omitted) and, worse, it doubled our
        # request rate right when the real problem turned out to be rate
        # limiting. Back to one clean request per date.
        params = dict(PARAMS_BASE, date=date_str, season_year=year)

        day_games = []
        html = ""
        # CHANGED: retry with increasing backoff if we get rate-limited.
        for attempt in range(4):
            try:
                r = session.get(BASE, params=params, headers=HEADERS, timeout=20)
                r.raise_for_status()
                html = r.text
            except Exception as e:
                print(f"[{i}/{len(all_dates)}] {date_str}: request FAILED ({e})")
                break

            # CHANGED: fingerprint this response by a hash of its content.
            # If we've seen this EXACT same page before for a DIFFERENT
            # date, the server isn't actually giving us that date's data —
            # it's handing back a cached/generic/rate-limited page instead.
            # (This is exactly what happened in your debug files: 5 pages
            # for 5 different years, byte-for-byte identical.)
            content_hash = hash(html)
            first_seen_date = seen_response_hashes.setdefault(content_hash, date_str)
            is_suspicious_repeat = (first_seen_date != date_str)

            if is_suspicious_repeat:
                wait = 5 * (attempt + 1)  # 5s, 10s, 15s, 20s
                print(f"[{i}/{len(all_dates)}] {date_str}: got the same page "
                      f"already seen for {first_seen_date} — looks rate-limited. "
                      f"Backing off {wait}s and retrying (attempt {attempt+1}/4)...")
                time.sleep(wait)
                continue  # retry this same date

            day_games = parse_day(html, date_str)
            break  # got a genuine, non-repeated response — stop retrying

        all_games.extend(day_games)
        print(f"[{i}/{len(all_dates)}] {date_str}: {len(day_games)} games")

        if len(day_games) == 0:
            zero_game_days.append(date_str)
            has_links = "/match/detail/" in html
            if not has_links:
                fname = f"debug_page_{date_str}.html"
                with open(fname, "w", encoding="utf-8") as dbg:
                    dbg.write(html)
                print(f"    -> No 'match/detail' links found. Saved to {fname}.")
            else:
                # CHANGED: this is the new, real case we need to diagnose —
                # the page DOES contain match links (so it's not blocked,
                # not rate-limited, not a real off-day), but parse_day()'s
                # card-detection couldn't extract anything from them. Save
                # a small HTML snippet around the FIRST such link so we can
                # see the actual markup and fix parse_day() against reality
                # instead of guessing again.
                soup = BeautifulSoup(html, "html.parser")
                first_link = soup.select_one('a[href*="/match/detail/"]')
                year_str = date_str[:4]
                already_for_year = snippets_saved_by_year.get(year_str, 0)
                should_save = (date_str in MUST_DEBUG_DATES) or (already_for_year < 2)
                if first_link is not None and should_save:
                    fname = f"debug_snippet_{date_str}.html"
                    node = first_link
                    for _ in range(8):
                        if node.parent is None:
                            break
                        node = node.parent
                    with open(fname, "w", encoding="utf-8") as dbg:
                        dbg.write(str(node.prettify()))
                    print(f"    -> Found match/detail link(s) but couldn't parse "
                          f"a game out of them. Saved surrounding HTML to {fname}.")
                    snippets_saved_by_year[year_str] = already_for_year + 1

        # CHANGED: longer, slightly randomized pause between dates (was a
        # flat 0.5s) — less likely to trip a requests-per-minute limit.
        time.sleep(1.5 + 0.5 * (i % 3))

    if zero_game_days:
        print(f"\n{len(zero_game_days)} of {len(all_dates)} days returned 0 games: "
              f"{zero_game_days[:10]}{'...' if len(zero_game_days) > 10 else ''}")
        print("If this is still a lot after the rate-limit backoff above, some of "
              "these may be genuine off-days (no Boys games that day, even if "
              "Girls/3x3 games were on) — check a couple of the debug_page_<date>.html "
              "files: if they show real DBL content with an empty games list, that's "
              "a real off-day; if they're all identical to each other again, we're "
              "still being rate-limited and need longer waits between requests.")

    if not all_games:
        print("No games parsed — dbl.id's markup may have changed; "
              "inspect one page's HTML and adjust parse_day().")
        return

    # Derived columns for the ML target: which team won, and by how much.
    # (Not real "home/away" — this is a neutral-venue tournament, so these
    # are really just "team 1 / team 2" as listed on the page.)
    for g in all_games:
        g["winner"] = g["home_team"] if g["home_score"] > g["away_score"] else g["away_team"]
        g["margin"] = abs(g["home_score"] - g["away_score"])

    fieldnames = ["game_id", "date", "season_year", "series", "region", "category",
                  "round", "venue", "time_wib", "home_team", "away_team",
                  "home_score", "away_score", "winner", "margin", "source_url"]

    with open("dbl_south_region_boys_games.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_games)

    print(f"\nWrote {len(all_games)} Boys games to dbl_south_region_boys_games.csv")


if __name__ == "__main__":
    main()

=== scrape_dbl_south_region.py — v5 (per-year + guaranteed debug snippets) ===

Live discovery found nothing new (expected — dbl.id's date tabs are JS-rendered and won't show up here). Using the 117 known verified dates.
Found 117 match days to scrape.
[1/117] 2019-08-31: 4 games
[2/117] 2019-09-01: 8 games
[3/117] 2019-09-02: 6 games
[4/117] 2019-09-03: 6 games
[5/117] 2019-09-04: 6 games
[6/117] 2019-09-05: 0 games
    -> Found match/detail link(s) but couldn't parse a game out of them. Saved surrounding HTML to debug_snippet_2019-09-05.html.
[7/117] 2019-09-06: 0 games
    -> Found match/detail link(s) but couldn't parse a game out of them. Saved surrounding HTML to debug_snippet_2019-09-06.html.
[8/117] 2019-09-07: 7 games
[9/117] 2019-09-08: 8 games
[10/117] 2019-09-09: 5 games
[11/117] 2019-09-10: 5 games
[12/117] 2019-09-11: 6 games
[13/117] 2019-09-14: 4 games
[14/117] 2019-09-15: 4 games
[15/117] 2019-09-17: 4 games
[16/117] 2019-09-18: 0 games
[17/117] 2019-09-20: 4 games
[18